# Swiss Health Insurance — Premium Region Classification
## HEC Lausanne | Machine Learning in Business Analytics | 2026

### Project Overview
Switzerland's health insurance system (LAMal) divides cantons into up to 3 premium
regions, frozen since 2004. This notebook applies a full ML pipeline to evaluate
whether current socio-demographic profiles still align with the official classification.

**Research question:** Can ML predict the official premium region of a Swiss commune
from its socio-demographic characteristics — and detect communes potentially
penalized by the frozen system?

---
## PART I — Data Pipeline
### Cell 1 — Imports

In [4]:
# Imports packages---------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from openpyxl import load_workbook

### Cell 2 — Paths & Configuration

We define the paths to the four raw data files and verify they exist before
proceeding. We also define the list of single-region cantons (CH0), which require
special handling in the join key construction (see Cell 3).

In [5]:
# Paths ----------------------------------------------------------------
BASE       = r'C:\Users\mahde\OneDrive\Desktop\ML_Project\data' + os.sep
OUTPUT_DIR = r'C:\Users\mahde\OneDrive\Desktop\ML_Project\outputs' + os.sep
OUTPUT_CSV = OUTPUT_DIR + 'swiss_communes_ml_ready.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# File paths ------------------------------------------------
PATH_REGIONS = BASE + 'Anhang EDI Ver. über die PrReg_Mut_2026.xlsx'
PATH_OFS = BASE + 'data.xlsx'
PATH_PRIMES25 = BASE + 'Prämien_CH_2025.xlsx'
PATH_PRIMES26 = BASE + 'Prämien_CH_2026.xlsx'

# Single-region cantons that appear as CH0 in premium files ---------------------------------------
CH0_CANTONS = ['AG','AI','AR','BS','GE','GL','JU','NE','NW','OW','SO','SZ','TG','UR','ZG']

print("All paths set up successfully.")


All paths set up successfully.


### Cell 3 — Load OFSP Region Labels

We parse the official OFSP ordinance (valid 01.01.2026) to extract the premium region
label for each commune. Each commune is identified by its BFS number.

**join_key logic:** Premium files use PR-REG CH0 for single-region cantons and
CH1/2/3 for multi-region zones. We build a composite key `canton_H{suffix}` to keep
each market context strictly separate — e.g. GE_H0 (Geneva, CHF 739) must never be
merged with BE_H1 (Berne zone 1, CHF 657).

In [6]:
def load_region_labels(filepath):
    wb = load_workbook(filepath, read_only=True, data_only=True)
    ws = wb['Anhang EDI Ver. über die PR']
    rows = []
    for row in ws.iter_rows(values_only=True):
        if (row[1] is not None and isinstance(row[1], (int, float))
                and row[0] and len(str(row[0]).strip()) <= 4):
            canton = str(row[0]).strip()
            bfs    = int(row[1])
            name   = str(row[2]).strip() if row[2] else ''
            region = int(row[3]) if row[3] else None
            if canton and region:
                join_key = f'{canton}_H0' if canton in CH0_CANTONS else f'{canton}_H{region}'
                rows.append({
                    'canton': canton, 'bfs_nr': bfs,
                    'commune': name, 'premium_region': region,
                    'join_key': join_key
                })
    df = pd.DataFrame(rows)
    print(f' {len(df)} communes loaded')
    print(f'   Class distribution: {df["premium_region"].value_counts().sort_index().to_dict()}')
    return df

df_labels = load_region_labels(PATH_REGIONS)
df_labels.head(3)

 1503 communes loaded
   Class distribution: {1: 332, 2: 824, 3: 347}


,canton,bfs_nr,commune,premium_region,join_key
0,BE,301,Aarberg,2,BE_H2
1,BE,321,Aarwangen,3,BE_H3
2,BE,561,Adelboden,3,BE_H3


### Cell 4 — Load Commune Fusions (Mutations)

Between the OFS 2024 reference and the OFSP 2026 labels, 22 communes merged.
We build a fusion map {old_BFS → new_BFS} from the Mutations sheet to ensure
all communes are correctly matched when we join the datasets later.

In [7]:
def load_mutations(filepath):
    wb = load_workbook(filepath, read_only=True, data_only=True)
    ws = wb['Mutationen']
    fusion_map = {}
    for row in ws.iter_rows(values_only=True):
        if not isinstance(row[0], (int, float)):
            continue
        mutation = str(row[3]).strip() if row[3] else ''
        if mutation.lower().startswith('fus'):
            parts = mutation.split()
            if len(parts) >= 2 and parts[1].isdigit():
                fusion_map[int(row[0])] = int(parts[1])
    print(f' {len(fusion_map)} commune fusions found')
    return fusion_map

fusion_map = load_mutations(PATH_REGIONS)

 22 commune fusions found


### Cell 5 — Load OFS Socio-demographic Features

#### Data Source
The Swiss Federal Statistical Office (OFS) publishes commune-level statistics
via the Swiss Stats Map Explorer, with reference date 01.01.2024. The raw Excel
file contains 2,131 communes and 12 columns.

#### Column Renaming
The original column names are long French strings prone to encoding issues.
We rename them by position, which is safer and more robust.

#### Data Quality Issues
Three data quality issues are handled:

1. **Header rows**: The Excel file contains 3 metadata rows before the actual
   data. We use `header=3` to skip them and start reading from the correct row.

2. **Confidential values**: The OFS masks values for small communes using
   `'*'` and `'N/A - secret statistique'` to protect statistical
   confidentiality. These are replaced with `NaN` before numeric conversion.

3. **Type coercion**: Because of the confidential markers, pandas reads numeric
   columns as `object` (string). We force conversion with
   `pd.to_numeric(..., errors='coerce')`.

#### Commune Fusions
We apply the fusion map built in Cell 4 to align BFS numbers with the
OFSP 2026 reference. When two communes merged, their old BFS number is
replaced by the new one.

#### Derived Features
Three additional features are engineered from the raw data:
- `log_pop_density`: log-transform of density to reduce right skew
- `is_urban`: binary flag, 1 if density > 1,000 hab/km²
- `pct_large_hh`: sum of 4-person and 5+-person households
- `pct_small_hh`: sum of 1-person and 2-person households

#### Missing Value Imputation
Remaining NaN values (after replacing confidential markers) are imputed
using the **column median**, which is robust to the right-skewed distributions
observed in density and social assistance variables.

#### Result
2,131 communes with 0 missing values, ready to be merged with the OFSP labels.
Note: the final dataset will contain 1,503 communes after the inner join with
the OFSP region labels (Cell 7).

In [8]:
def load_ofs_stats(filepath, fusion_map):
    df = pd.read_excel(filepath, header=3)

    # Rename columns by position (safer than matching long French strings)
    df.columns = [
    'bfs_nr', 'commune_ofs', 'pop_density_2024',
    'pct_hh_1p', 'pct_hh_2p', 'pct_hh_3p',
    'pct_hh_4p', 'pct_hh_5p_plus',
    'social_assistance_rate', 'sa_estimated',
    'social_assistance_count', 'sa_count_est'
    ]

    # Keep only relevant columns
    keep = ['bfs_nr', 'commune_ofs', 'pop_density_2024',
            'pct_hh_1p', 'pct_hh_2p', 'pct_hh_3p',
            'pct_hh_4p', 'pct_hh_5p_plus',
            'social_assistance_rate',
            'social_assistance_count']
    df = df[keep].copy()

    # Replace all confidential/missing value markers with NaN
    df = df.replace(['*', 'N/A - secret statistique'], np.nan)

    # Force numeric conversion
    for col in df.columns:
        if col not in ['commune_ofs']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Apply commune fusions
    df['bfs_nr'] = df['bfs_nr'].replace(fusion_map)

    # Derived features
    df['log_pop_density'] = np.log1p(df['pop_density_2024'])
    df['is_urban']        = (df['pop_density_2024'] > 1000).astype(int)
    df['pct_large_hh']   = df['pct_hh_4p'] + df['pct_hh_5p_plus']
    df['pct_small_hh']   = df['pct_hh_1p'] + df['pct_hh_2p']

    # Median imputation for missing values
    for col in df.select_dtypes(include=np.number).columns:
        df[col] = df[col].fillna(df[col].median())

    print(f' OFS stats: {len(df)} communes | {df.isnull().sum().sum()} missing values')
    return df

df_ofs = load_ofs_stats(PATH_OFS, fusion_map)
df_ofs.head(3)

 OFS stats: 2131 communes | 0 missing values


,bfs_nr,commune_ofs,pop_density_2024,pct_hh_1p,pct_hh_2p,pct_hh_3p,pct_hh_4p,pct_hh_5p_plus,social_assistance_rate,social_assistance_count,log_pop_density,is_urban,pct_large_hh,pct_small_hh
0,1,Aeugst am Albis,252.2,28.9,38.2,13.2,13.9,5.8,1.2,23.0,5.534180,0,19.7,67.1
1,10,Obfelden,790.6,29.3,34.7,13.0,16.5,6.4,2.1,126.0,6.674056,0,22.9,64.0
2,100,Stadel,190.5,32.5,36.0,13.0,13.5,5.0,1.0,24.0,5.254888,0,18.5,68.5


### Cell 6 — Load Premium Market Data (2025 & 2026)

We load the official premium price files from opendata.swiss. We filter on a
standard adult profile: age class AKL-ERW, TAR-BASE model, FRAST1 (CHF 300
franchise), base insurer only (isBaseF = 1).

We then compute aggregate statistics per join_key: average, median, standard
deviation, 10th and 90th percentiles, IQR, number of insurers, and the
year-over-year premium increase.

We exclude cantons ZE and ZR which contain implausible prices.

In [9]:
def load_premiums(path25, path26, ch0_cantons):
    dfs = {}
    for year, path in [(2025, path25), (2026, path26)]:
        df = pd.read_excel(path)

        # Standardize column names to lowercase
        df.columns = df.columns.str.lower().str.strip()

        # Filter standard adult profile
        df = df[
            (df['altersklasse']  == 'AKL-ERW') &
            (df['tarif']         == 'BASE') &
            (df['franchisestufe']== 'FRAST1') &
            (df['isbasef']       == 1)
        ].copy()

        # Exclude invalid cantons
        df = df[~df['kanton'].isin(['ZE', 'ZR'])]

        # Build join_key
        df['region_suffix'] = df['region'].astype(str).str[-1]
        df['join_key'] = df.apply(
            lambda r: f"{r['kanton']}_H0" if r['kanton'] in ch0_cantons
                      else f"{r['kanton']}_H{r['region_suffix']}",
            axis=1
        )

        dfs[year] = df
        print(f' Premiums {year}: {len(df)} rows after filtering')

    return dfs[2025], dfs[2026]

df_p25, df_p26 = load_premiums(PATH_PRIMES25, PATH_PRIMES26, CH0_CANTONS)

 Premiums 2025: 2220 rows after filtering
 Premiums 2026: 2046 rows after filtering


### Cell 7 — Aggregate Premium Statistics

For each canton-region market (join_key), we compute descriptive statistics
across all insurers: mean, median, standard deviation, 10th and 90th
percentiles, IQR, and number of active insurers. We also compute the
year-over-year premium increase from 2025 to 2026.

These aggregated features capture the market-level premium structure
for each region, which will be used in Experiment A (full features).

In [10]:
def aggregate_premiums(df25, df26):
    # Aggregate 2026 premiums by join_key
    agg26 = df26.groupby('join_key')['prämie'].agg(
        avg_premium_2026    = 'mean',
        median_premium_2026 = 'median',
        std_premium_2026    = 'std',
        p10_premium_2026    = lambda x: x.quantile(0.10),
        p90_premium_2026    = lambda x: x.quantile(0.90),
        iqr_premium_2026    = lambda x: x.quantile(0.75) - x.quantile(0.25),
        n_insurers_2026     = 'count'
    ).reset_index()

    # Aggregate 2025 premiums by join_key
    agg25 = df25.groupby('join_key')['prämie'].agg(
        avg_premium_2025 = 'mean'
    ).reset_index()

    # Merge and compute YoY increase
    agg = agg26.merge(agg25, on='join_key', how='left')
    agg['premium_increase_pct'] = (
        (agg['avg_premium_2026'] - agg['avg_premium_2025'])
        / agg['avg_premium_2025'] * 100
    ).round(2)

    print(f' Premium stats: {len(agg)} join_keys')
    print(agg[['join_key', 'avg_premium_2026', 'premium_increase_pct']].head(5))
    return agg

df_premiums = aggregate_premiums(df_p25, df_p26)

 Premium stats: 42 join_keys
  join_key  avg_premium_2026  premium_increase_pct
0    AG_H0        551.139815                  4.69
1    AI_H0        444.102174                  3.26
2    AR_H0        531.410870                  4.11
3    BE_H1        657.375962                  3.02
4    BE_H2        590.631731                  3.06


### Cell 8 — Merge All Datasets

We perform two successive inner joins:
1. OFSP labels (1,503 communes) ← joined with → OFS features (2,131 communes)
   on `bfs_nr`. The inner join keeps only communes present in both datasets.
2. Result ← joined with → premium statistics (42 join_keys) on `join_key`.

The final dataset contains 1,503 communes with all features and the target
variable `premium_region`.

In [11]:
def merge_all(df_labels, df_ofs, df_premiums):
    # Step 1: merge labels with OFS features
    df = df_labels.merge(df_ofs, on='bfs_nr', how='inner')
    print(f'After labels + OFS merge: {len(df)} communes')

    # Step 1: merge labels with OFS features
    df = df_labels.merge(df_ofs, on='bfs_nr', how='inner')
    
    # Remove duplicates created by commune fusions
    df = df.drop_duplicates(subset='bfs_nr', keep='first')
    print(f'After labels + OFS merge: {len(df)} communes')

    # Step 2: merge with premium statistics
    df = df.merge(df_premiums, on='join_key', how='left')
    print(f'After premium merge:      {len(df)} communes')
    print(f'Missing values:           {df.isnull().sum().sum()}')
    print(f'Columns:                  {len(df.columns)}')
    return df

df_final = merge_all(df_labels, df_ofs, df_premiums)
df_final.head(3)

After labels + OFS merge: 1516 communes
After labels + OFS merge: 1503 communes
After premium merge:      1503 communes
Missing values:           0
Columns:                  27


,canton,bfs_nr,commune,premium_region,join_key,commune_ofs,pop_density_2024,pct_hh_1p,pct_hh_2p,pct_hh_3p,...,pct_small_hh,avg_premium_2026,median_premium_2026,std_premium_2026,p10_premium_2026,p90_premium_2026,iqr_premium_2026,n_insurers_2026,avg_premium_2025,premium_increase_pct
0,BE,301,Aarberg,2,BE_H2,Aarberg,582.6,37.9,35.4,11.5,...,73.3,590.631731,587.25,32.595971,549.37,629.59,45.3,52,573.069643,3.06
1,BE,321,Aarwangen,3,BE_H3,Aarwangen,486.2,33.9,35.6,12.4,...,69.5,554.154808,554.30,32.534700,520.53,589.13,48.8,52,536.474107,3.30
2,BE,561,Adelboden,3,BE_H3,Adelboden,39.0,34.9,36.0,11.2,...,70.9,554.154808,554.30,32.534700,520.53,589.13,48.8,52,536.474107,3.30


### Cell 9 — Export ML-Ready Dataset

We export the final merged dataset to a CSV file. This file will be used
as the starting point for all subsequent analyses (EDA, unsupervised and
supervised learning) to avoid re-running the full pipeline each time.

In [12]:
df_final.to_csv(OUTPUT_CSV, index=False)
print(f' Dataset exported: {OUTPUT_CSV}')
print(f'   Shape: {df_final.shape}')
print(f'   Class distribution:')
print(df_final['premium_region'].value_counts().sort_index())

 Dataset exported: C:\Users\mahde\OneDrive\Desktop\ML_Project\outputs\swiss_communes_ml_ready.csv
   Shape: (1503, 27)
   Class distribution:
premium_region
1    332
2    824
3    347
Name: count, dtype: int64
